# Week 6 – Product Pricer Challenge

**Goal:** Build a model that predicts how much a product costs from its description.

| Step | What we do |
|------|------------|
| 1 | Load data from HuggingFace Hub |
| 2 | Explore the dataset (price distribution, categories, text length) |
| 3 | Establish a baseline with GPT-4o-mini |
| 4 | Prepare JSONL fine-tuning data |
| 5 | Upload & launch a fine-tuning job on OpenAI |
| 6 | Evaluate the fine-tuned model vs. baseline |
| 7 | (Optional) Enhanced prompts & second fine-tune |

In [ ]:
import os
import re
import sys
import json
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from dotenv import load_dotenv
from openai import OpenAI
from huggingface_hub import login

from pricer.items import Item
from pricer.evaluator import Tester

load_dotenv(override=True)
hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)

openai = OpenAI()

%matplotlib inline
print("All imports loaded.")

## 1 – Load the dataset

We pull the **lite** dataset from HuggingFace (20 000 train / 1 000 val / 1 000 test).  
Set `LITE_MODE = False` to use the full 800 000-item dataset instead.

In [ ]:
LITE_MODE = True

HF_USER = "williampepple1"
dataset_name = f"{HF_USER}/items_lite" if LITE_MODE else f"{HF_USER}/items_full"

train, val, test = Item.from_hub(dataset_name)

print(f"Train : {len(train):,}")
print(f"Val   : {len(val):,}")
print(f"Test  : {len(test):,}")
print(f"\nSample item: {train[0]}")
print(f"Prompt preview:\n{train[0].test_prompt()[:300]}...")

## 2 – Explore the data

In [ ]:
prices = [item.price for item in train]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(prices, bins=range(0, 1000, 10), color="blueviolet", rwidth=0.7)
axes[0].set_title(f"Price distribution (avg ${np.mean(prices):,.1f})")
axes[0].set_xlabel("Price ($)")
axes[0].set_ylabel("Count")

category_counts = Counter(item.category for item in train)
cats = list(category_counts.keys())
cnts = [category_counts[c] for c in cats]
axes[1].barh(cats, cnts, color="goldenrod")
axes[1].set_title("Items per category")
axes[1].set_xlabel("Count")
for i, v in enumerate(cnts):
    axes[1].text(v + 50, i, f"{v:,}", va="center")

plt.tight_layout()
plt.show()

print(f"Price range: ${min(prices):.2f} – ${max(prices):.2f}")
print(f"Categories : {len(category_counts)}")

## 3 – Helper functions

In [ ]:
SYSTEM_PROMPT = "You estimate prices of items. Reply only with the price, no explanation"


def get_price(text):
    """Extract a numeric price from free-form text."""
    text = str(text).replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", text)
    return float(match.group()) if match else 0


def messages_for(item):
    """Build chat messages for inference (no answer)."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item.test_prompt()},
    ]


def messages_with_answer(item):
    """Build chat messages with the ground-truth price (for fine-tuning)."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item.test_prompt()},
        {"role": "assistant", "content": f"Price is ${item.price:.2f}"},
    ]


print("Helper functions ready.")

## 4 – Baseline: GPT-4o-mini (no fine-tuning)

We evaluate the stock model on 200 test items using the course `Tester` class.

In [ ]:
def gpt_4o_mini_baseline(item):
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages_for(item),
        seed=42,
        max_tokens=8,
    )
    return get_price(response.choices[0].message.content)


baseline_tester = Tester(gpt_4o_mini_baseline, test, title="GPT-4o-mini baseline", size=200)
baseline_tester.run()

## 5 – Prepare fine-tuning JSONL files

OpenAI recommends at least 50–100 examples; we use the full training and validation splits.

In [ ]:
def write_jsonl(items, filepath):
    with open(filepath, "w") as f:
        for item in items:
            line = {"messages": messages_with_answer(item)}
            f.write(json.dumps(line) + "\n")


TRAIN_JSONL = "fine_tune_train.jsonl"
VAL_JSONL = "fine_tune_val.jsonl"

write_jsonl(train, TRAIN_JSONL)
write_jsonl(val, VAL_JSONL)

print(f"Wrote {len(train):,} training examples to {TRAIN_JSONL}")
print(f"Wrote {len(val):,} validation examples to {VAL_JSONL}")

## 6 – Upload files & launch the fine-tuning job

In [ ]:
with open(TRAIN_JSONL, "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

with open(VAL_JSONL, "rb") as f:
    val_file = openai.files.create(file=f, purpose="fine-tune")

print(f"Training file   : {train_file.id}")
print(f"Validation file : {val_file.id}")

In [ ]:
fine_tuning_job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4o-mini",
    seed=42,
    hyperparameters={"n_epochs": 1},
    suffix="pricer",
)

job_id = fine_tuning_job.id
print(f"Fine-tuning job launched: {job_id}")
print("Monitor at: https://platform.openai.com/finetune")

## 7 – Monitor the job

Re-run this cell until status shows **succeeded**.

In [ ]:
status = openai.fine_tuning.jobs.retrieve(job_id)
print(f"Status : {status.status}")
print(f"Model  : {status.fine_tuned_model or '(not ready)'}")

events = openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=5)
for ev in events.data:
    print(f"  {ev.created_at}  {ev.message}")

## 8 – Evaluate the fine-tuned model

In [ ]:
fine_tuned_model = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

if not fine_tuned_model:
    print("Fine-tuned model not ready yet. Re-run the monitoring cell above.")
else:
    print(f"Fine-tuned model: {fine_tuned_model}")

    def gpt_fine_tuned(item):
        response = openai.chat.completions.create(
            model=fine_tuned_model,
            messages=messages_for(item),
            seed=42,
            max_tokens=8,
        )
        return get_price(response.choices[0].message.content)

    ft_tester = Tester(gpt_fine_tuned, test, title="GPT-4o-mini (fine-tuned)", size=200)
    ft_tester.run()

## 9 – (Optional) Enhanced prompts & second fine-tune

A richer system prompt that gives the model more context about pricing.

In [ ]:
ENHANCED_SYSTEM_PROMPT = (
    "Role: You are a retail price estimator.\n"
    "Market: United States; Currency: USD.\n"
    "Scope: Predict the most likely new retail price. "
    "Ignore taxes, shipping, coupons, bundles, used/renewed.\n"
    "Output: Only a number with two decimals (e.g. 129.99). No $ sign. No words.\n"
    "Think silently; do not reveal reasoning."
)


def enhanced_messages_for(item):
    return [
        {"role": "system", "content": ENHANCED_SYSTEM_PROMPT},
        {"role": "user", "content": item.test_prompt()},
    ]


def enhanced_messages_with_answer(item):
    return [
        {"role": "system", "content": ENHANCED_SYSTEM_PROMPT},
        {"role": "user", "content": item.test_prompt()},
        {"role": "assistant", "content": f"{item.price:.2f}"},
    ]


def write_jsonl_v2(items, filepath):
    with open(filepath, "w") as f:
        for item in items:
            line = {"messages": enhanced_messages_with_answer(item)}
            f.write(json.dumps(line) + "\n")


TRAIN_V2 = "fine_tune_train_v2.jsonl"
VAL_V2 = "fine_tune_val_v2.jsonl"

write_jsonl_v2(train, TRAIN_V2)
write_jsonl_v2(val, VAL_V2)

print(f"Enhanced JSONL files written: {TRAIN_V2}, {VAL_V2}")

In [ ]:
with open(TRAIN_V2, "rb") as f:
    train_file_v2 = openai.files.create(file=f, purpose="fine-tune")

with open(VAL_V2, "rb") as f:
    val_file_v2 = openai.files.create(file=f, purpose="fine-tune")

fine_tuning_job_v2 = openai.fine_tuning.jobs.create(
    training_file=train_file_v2.id,
    validation_file=val_file_v2.id,
    model="gpt-4o-mini",
    seed=42,
    hyperparameters={"n_epochs": 1},
    suffix="pricer-v2",
)

job_id_v2 = fine_tuning_job_v2.id
print(f"Enhanced fine-tuning job launched: {job_id_v2}")

In [ ]:
status_v2 = openai.fine_tuning.jobs.retrieve(job_id_v2)
print(f"Status : {status_v2.status}")
print(f"Model  : {status_v2.fine_tuned_model or '(not ready)'}")

In [ ]:
enhanced_model = openai.fine_tuning.jobs.retrieve(job_id_v2).fine_tuned_model

if not enhanced_model:
    print("Enhanced model not ready yet. Re-run the status cell above.")
else:
    print(f"Enhanced model: {enhanced_model}")

    def gpt_enhanced(item):
        response = openai.chat.completions.create(
            model=enhanced_model,
            messages=enhanced_messages_for(item),
            seed=42,
            max_tokens=8,
        )
        return get_price(response.choices[0].message.content)

    enhanced_tester = Tester(gpt_enhanced, test, title="GPT-4o-mini (enhanced ft)", size=200)
    enhanced_tester.run()

## 10 – Summary

| Model | What it does |
|-------|--------------|
| GPT-4o-mini baseline | Stock model, zero-shot pricing |
| GPT-4o-mini fine-tuned (v1) | Same simple prompt, trained on our data |
| GPT-4o-mini fine-tuned (v2) | Enhanced system prompt with market context |

Compare the error charts above to see how fine-tuning improves accuracy.